<a href="https://colab.research.google.com/github/HoriaCluj/Machine-Learning/blob/main/Binary_Classifier_Neural_Network.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### **Binary Classifier Neural Network**

Building a Neural Network in PyTorch that will predict whether the next trading day will be an upward or downward closing day, based on past data I will train it on.

**Start by Importing the Modules**

In [35]:
import yfinance as yf
import pandas as pd
import numpy as np
import torch

Using `yfinance`, import the stock data of the past 25 years, which will be stored in the `df` dataframe.

In [36]:
df = yf.download('SPY', period='30y')
df

/tmp/ipython-input-2012554920.py:1: FutureWarning: YF.download() has changed argument auto_adjust default to True
  df = yf.download('SPY', period='30y')
[*********************100%***********************]  1 of 1 completed


Price,Close,High,Low,Open,Volume
Ticker,SPY,SPY,SPY,SPY,SPY
Date,,,,,
1996-02-05,38.111271,38.166962,37.684306,37.739997,295300
1996-02-06,38.473259,38.510386,38.074140,38.129831,301600
1996-02-07,38.696030,38.696030,38.426856,38.463983,585100
1996-02-08,39.113708,39.150836,38.566079,38.640334,1526600
1996-02-09,39.104431,39.429296,38.918794,39.095149,804600
...,...,...,...,...,...
2026-01-30,691.969971,694.210022,687.119995,691.789978,101835100
2026-02-02,695.409973,696.929993,689.419983,689.580017,79286500


In [37]:
df = yf.download('SPY', period='30y')
df = df[['Open', 'High', 'Low', 'Close', 'Volume']]
df.head()

/tmp/ipython-input-3004003279.py:1: FutureWarning: YF.download() has changed argument auto_adjust default to True
  df = yf.download('SPY', period='30y')
[*********************100%***********************]  1 of 1 completed


Price,Open,High,Low,Close,Volume
Ticker,SPY,SPY,SPY,SPY,SPY
Date,,,,,
1996-02-05,37.739986,38.166951,37.684295,38.111259,295300
1996-02-06,38.129823,38.510379,38.074132,38.473251,301600
1996-02-07,38.463972,38.696018,38.426845,38.696018,585100
1996-02-08,38.640327,39.150828,38.566072,39.113701,1526600
1996-02-09,39.095149,39.429296,38.918794,39.104431,804600


### **Training the Model**

* In this section, we are turning all 25 years of historical stock data into samples of 10 days, in order to train our model.
* Our model will learn from (many samples of) the last 10 days of trading data to predict the next day's movement (up or down).
* Each window of 10 days becomes a training sample. The Neural Network will learn from a set of features (OHLCV) during those 10 days and how they affect the next day's movement (label: 1 UP, 0 DOWN)

Define the **Create Sequences Function**


* `X` represent the input sequences (past windows of 10 days)
* `y` represents the output labels (1 = Up; 0 = Down)

For-Loop Explanation:

* We are looping from `iteration = 0` to the latest data we can use to train, meaning the length of our dataset `-` a window (of 10 days)
* `X_Seq` is the past 10 days of features, an array starting from the current iteration until the current iteration + 10 days (window)
* `open_day_eleven` retrieves the open of the eleventh day (prediction day), by indexing the 0th column of the dataframe as `data[0] = Open`
* `close_day_eleven` retrieves the close of the eleventh day (prediction day), by indexing the 3rd column of the dataframe as `data[3] = Close`
* `label = int(close_day11 > open_day11)` transforms a True/False output to a binary `True = 1`, and `False = 2`, then stores it in the label variable.

* `X.append(X_seq)` adds the samples of 10 days at each iteration to the X variable.
* `y.append(labels)` adds the labels to y variable, at each iteration.


The `create_sequences` function will therefore return the following:

* `np.array(X)` creates a NumPy array with the sample sequences (of 10 days)
* `np.array(y)` creates a NumPy array with the obtained labels.


In [38]:
def create_sequences(data, window=10):
  X = []
  y = []

  for i in range(len(data) - window -1):
    X_seq = data[i: i+window]

    open_day_eleven = data[i + window][0]
    close_day_eleven = data[i + window][3]

    label = int(close_day_eleven > open_day_eleven)

    X.append(X_seq)
    y.append(label)

  return np.array(X), np.array(y)

**Prepare the Data**

`data = df.values` transforms the Pandas DataFrame to a NumPy array, we essentially transform the nice-looking table to an array of numbers (features).

`X, y = create_sequences(data)`- we pass the NumPy array created above through the `create_sequences()` function

In [39]:
data = df.values
X, y = create_sequences(data)

### **Split Data and Normalize Features**

`train_test_split` splits the data into **training** and **testing** sets

`StandardScaler` standardizes your features (OHLCV).

* Normalizing infers that each feature will have `Mean = 0` and `Standard Deviation = 1`
* `Standardized x = (x - μ) / σ`
* This allows every feature to be between -1 and 1, centered around 0, which is much easier for the Neural Network to learn.

`TensorDataset` bundles `(X, y)` pairs together

`DataLoader` helps automatically batch the samples

In [40]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import torch
from torch.utils.data import TensorDataset, DataLoader

### **Normalize Features Values (OHLCV)**

In [41]:
scaler = StandardScaler()

`X` is initially of shape `(samples, window, features)`

This can be understood via the `create_sequences()` function.


1.   `data` is a NumPy array of shape Days | Features
  * see `df.head()` for reference
2. `X_seq = data[iteration; iteration + window]` means we are slicing the data in 10-day windows.
  * Each `X_seq` is therefore of shape `(window, features)` -> (10, 5)
3. We are appending many `X_seq` to the list `X = []`
  * Hence, `X = []` is of 3D shape -> (nr. samples, window, features)



However, `StandardScaler()` expects a **2D input**, of `[n_samples, n_features]`



1.   We create the `X_scaled` variable that will hold the **normalized version of X**.
2. `scaler.fit_transform()` is a **pre-built function to normalize the features** data.
3. As `scaler` expects a 2D input, **we reshape** `X` **to 2D by giving the input** `X.reshape(-1, X.shape[-1])`
  * Transforms the `X` shape temporarily from `(samples, window, features)` to `(samples * window, features)`
4. **After standardizing all features** across all days, you **reshape back** `X` to its original shape, using `.reshape(X.shape)`


In [42]:
X_scaled = scaler.fit_transform(X.reshape(-1, X.shape[-1]))
X_scaled = X_scaled.reshape(X.shape)
X_scaled

array([[[-0.93855259, -0.93803332, -0.93640486, -0.93601339,
         -1.01558435],
        [-0.93598188, -0.93577949, -0.93382041, -0.93362682,
         -1.01551437],
        [-0.9337784 , -0.93456118, -0.93148209, -0.93215814,
         -1.01236516],
        ...,
        [-0.92833088, -0.92980979, -0.92797456, -0.93032225,
         -1.01407251],
        [-0.93096278, -0.93157629, -0.92975904, -0.93191328,
         -1.00898378],
        [-0.93249306, -0.93419571, -0.9312975 , -0.93295368,
         -1.01213188]],

       [[-0.93598188, -0.93577949, -0.93382041, -0.93362682,
         -1.01551437],
        [-0.9337784 , -0.93456118, -0.93148209, -0.93215814,
         -1.01236516],
        [-0.93261546, -0.93157638, -0.93055907, -0.9294044 ,
         -1.00190666],
        ...,
        [-0.93096278, -0.93157629, -0.92975904, -0.93191328,
         -1.00898378],
        [-0.93249306, -0.93419571, -0.9312975 , -0.93295368,
         -1.01213188],
        [-0.93500261, -0.93657141, -0.93468193, 

In [43]:
X_scaled.shape, y.shape

((7539, 10, 5), (7539,))

### **Split into Training and Testing Sets**

Use the sci-kit library's `train_test_split(X, y, test_size)`

In [44]:
X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2)

Implement the Data Loader to avoid running the whole dataset at once through the training loop, but rather in small batches to optimize training.

In [45]:
# Convert to PyTorch tensors
X_train = torch.tensor(X_train, dtype=torch.float32)
y_train = torch.tensor(y_train, dtype=torch.float32)
X_test = torch.tensor(X_test, dtype=torch.float32)
y_test = torch.tensor(y_test, dtype=torch.float32)

# Create TensorDatasets
train_ds = TensorDataset(X_train, y_train)
test_ds = TensorDataset(X_test, y_test)

# Create DataLoaders
train_loader = DataLoader(train_ds, batch_size=64, shuffle=True)
test_loader = DataLoader(test_ds, batch_size=64)

In [46]:
X_train.shape, X_test.shape, y_train.shape, y_test.shape

(torch.Size([6031, 10, 5]),
 torch.Size([1508, 10, 5]),
 torch.Size([6031]),
 torch.Size([1508]))

### **Building the Binary Classification Neural Network**

We will implement two hidden layers, as it gives the model more flexibility to learn non-linear patterns.
1. **Layer 1** maps 50 raw inputs (5 features x 10 days) to 64 hidden-layer features. Each of the 64 output features is a weighted linear combination of the 50 raw inputs: `hidden_i = w1x1 + w2x2 + ... + w50x50`. We then apply the `ReLU(x) = max(0, x)` to introduce non-linearity.
2. **Layer 2** takes the 64 output features from the previous layer and transforms them into 32 new weighted combinations. The ReLU is then once again applied for a non-linear boundary.
3. **Output Layer** maps the 32 learned weighted combinations into one raw output, to which we apply the Sigmoid function to obtain a probability (0 to 1) for binary classification

The **`forward()`** method:
- Since `nn.Linear()` expects a 2-dimensional input; one vector per sample.
- `x.view(x.size(0), -1)` flattens the input from 3D to a 2D vector: e.g. `X_train.shape = (5020, 10, 5) -> (5020, 50)`
1. Pass the 50 (10x5) input features through the first layer, to which we apply the non-linear ReLU function
2. Pass the 64 learned weighted combinations through the second layer, to which we once again apply the ReLU transformation.
3. `self.sigmoid()` transforms the output value into a probability between 0 (Down Day) and 1 (Up Day).

In [47]:
from torch import nn

In [48]:
class BinaryClassification(nn.Module):
  def __init__(self):
    super().__init__()

    # Layer 1: Input Layer -> Hidden Layer
    self.f1 = nn.Linear(50, 64)
    self.relu1 = nn.ReLU()

    # Layer 2: Hidden Layer -> Hidden Layer
    self.f2 = nn.Linear(64, 32)
    self.relu2 = nn.ReLU()

    # Layer 3: Hidden Layer -> 1 Output Node
    self.out = nn.Linear(32, 1)
    self.sigmoid = nn.Sigmoid()

  def forward(self, x):
    # Flatten the Input Vector from 3D to 2d
    x = x.view(x.size(0), -1)

    # Forward Pass through the Neural Network
    x = self.relu1(self.f1(x))
    x = self.relu2(self.f2(x))
    x = self.sigmoid(self.out(x))
    return x

Instantiate the Binary Classification Model, the initial predictions will be probabilities calculated using random (initial guesses) of parameters as we have yet to train the model. First, we must convert the NumPy arrays to `torch.tensor`

In [49]:
model = BinaryClassification()
y_preds = model.forward(X_train)

y[:5], y_preds[:5], y_preds.shape

(array([0, 1, 1, 0, 0]),
 tensor([[0.5473],
         [0.4985],
         [0.5045],
         [0.5179],
         [0.5013]], grad_fn=<SliceBackward0>),
 torch.Size([6031, 1]))

### **Training & Testing Loop Construction**

Start by unsqueezing the `y_train` and `y_test` vector to match the dimension of `y_preds`, `unsqueeze` adds a dimension.

In [50]:
y_train, y_test = y_train.unsqueeze(1), y_test.unsqueeze(1)
y_test.shape

torch.Size([1508, 1])

Set up the Loss Function: **`nn.BCELoss()`**, since the sigmoid transformation is already implemented in our forward method.

In [51]:
loss_fn = nn.BCELoss()

Set up the **Optimizer SGD**: Stochastic Gradient Descent with a Learning Rate of 0.1

In [52]:
optimizer = torch.optim.SGD(model.parameters(), lr=0.1)

#### **Training Loop**
1. Forward Pass through the `BinaryClassification` model
2. Calculate the `BCE Loss`
3. Optimizer Zero Grad
4. Perform `Backpropagation`
5. Optimizer Step (Stochasic Gradient Descent)

In [53]:
epochs = 1000
best_loss = float('inf')
patience = 20 # Stop the Loop if there is no improvement in the test loss after 20 epochs
counter = 0
best_model = None

for epoch in range(epochs):
  model.train()

  # Training Loop
  # 1. Forward Pass & 2. BCE Loss Function
  for xb, yb in train_loader:
    y_final = model.forward(xb).squeeze()
    train_loss = loss_fn(y_final, yb)

    # 3. Optimizer Zero Grad
    optimizer.zero_grad()

    # 4. Backpropagation (Loss Function Gradient)
    train_loss.backward()

    # 5. Optimizer Step (SGD)
    optimizer.step()

  # Testing Loop
  model.eval()
  running_test_loss = 0.0

  with torch.no_grad():
    for xt, yt in test_loader:
      # 1. Forward Pass
      test_final = model.forward(xt).squeeze()

      # 2. Loss Function
      test_loss = loss_fn(test_final, yt)
      running_test_loss += test_loss.item()

    # Early stoppage
    if running_test_loss < best_loss:
      best_loss = running_test_loss
      counter = 0
      best_model = model.state_dict()
    else:
      counter +=1
      if counter >= patience:
        break

  if epoch % 1 == 0:
    print(f'Epoch: {epoch} | Training Loss: {train_loss} | Test Loss: {test_loss}')

# Restoring the Best Model
if best_model:
    model.load_state_dict(best_model)
with torch.no_grad():
    correct_total = 0
    total_samples = 0
    for xt, yt in test_loader:
        test_final = model(xt).squeeze()
        pred_classes = (test_final > 0.5).float()
        correct_total += (pred_classes == yt).sum().item()
        total_samples += yt.size(0)

final_hit_rate = correct_total / total_samples
print(final_hit_rate)

Epoch: 0 | Training Loss: 0.6988773941993713 | Test Loss: 0.7025668621063232
Epoch: 1 | Training Loss: 0.6950934529304504 | Test Loss: 0.7030866146087646
Epoch: 2 | Training Loss: 0.7094327211380005 | Test Loss: 0.7030305862426758
Epoch: 3 | Training Loss: 0.6832086443901062 | Test Loss: 0.7060462236404419
Epoch: 4 | Training Loss: 0.7103015780448914 | Test Loss: 0.7033848762512207
Epoch: 5 | Training Loss: 0.70024174451828 | Test Loss: 0.7004888653755188
Epoch: 6 | Training Loss: 0.6904458403587341 | Test Loss: 0.7013397216796875
Epoch: 7 | Training Loss: 0.6967769861221313 | Test Loss: 0.7042104601860046
Epoch: 8 | Training Loss: 0.7209140062332153 | Test Loss: 0.7019505500793457
Epoch: 9 | Training Loss: 0.6818451285362244 | Test Loss: 0.7031891345977783
Epoch: 10 | Training Loss: 0.671934962272644 | Test Loss: 0.7031010985374451
Epoch: 11 | Training Loss: 0.6821013689041138 | Test Loss: 0.7024708986282349
Epoch: 12 | Training Loss: 0.7100942134857178 | Test Loss: 0.70051109790802
E

In [54]:
with torch.inference_mode():
  y_preds = torch.round(model(X_test))

print("Reality: ", y_test[10:].squeeze(1))
print("Predictions: ", y_preds[10:].squeeze(1))

Reality:  tensor([0., 1., 1.,  ..., 1., 1., 0.])
Predictions:  tensor([0., 0., 1.,  ..., 1., 0., 1.])


Computing the **Confusion Matrix** to evaluate the predictive power of the model

In [55]:
from sklearn.metrics import confusion_matrix, classification_report

print(confusion_matrix(y_preds, y_test))
print(classification_report(y_preds, y_test, digits=3))

[[ 74  69]
 [627 738]]
              precision    recall  f1-score   support

         0.0      0.106     0.517     0.175       143
         1.0      0.914     0.541     0.680      1365

    accuracy                          0.538      1508
   macro avg      0.510     0.529     0.427      1508
weighted avg      0.838     0.538     0.632      1508



### **Predicting Tomorrow**

To conclude, we will use this classification neural network to predict whether tomorrow will be an **Up day** (Closing Price > Opening Price) or **Down Day** (Closing Price < Opening Price). Please note that this is only indicative, since the ~52% accuracy (hit rate) was achieved over 30 years of data, so over a small sample, accuracy may differ significantly.
- Take the most recent sequence of 10 days and use the model to predict the state of the eleventh day, which should correspond to **tomorrow**.
- Make sure to transform the sequence to a tensor before feeding it to the model, otherwise an int error will occur.

In [56]:
most_recent = X_scaled[-1]

model.eval()
with torch.inference_mode():
  most_recent = torch.tensor(most_recent, dtype=torch.float32).unsqueeze(0)
  tomorrow_prediction = torch.round(model(most_recent))

tomorrow_prediction = int(tomorrow_prediction)

1

#### **Concluding:**

In [ ]:
if tomorrow_prediction == 1:
  print("With an Expected Precision of ", final_hit_rate, "tomorrow will be an Up-day")